In [ ]:
from langchain_ollama import OllamaEmbeddings
from qdrant_client import QdrantClient
import os
from typing import Type
from dotenv import load_dotenv
load_dotenv()
QDRANT_URL = os.getenv("QDRANT_URL")
#client_qd = QdrantClient(url=QDRANT_URL)
client_qd = QdrantClient(path="./qdrant_data")
from pathlib import Path
from typing import List, Dict
from langchain_openai import OpenAIEmbeddings
import hashlib
import uuid
import docx2txt
from pypdf import PdfReader
from langchain_community.document_loaders import PyPDFLoader, Docx2txtLoader
from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance, VectorParams
from langchain_qdrant import QdrantVectorStore
import requests
from langchain.text_splitter import CharacterTextSplitter
try:
    from langchain_core.documents import Document # newer langchain
except Exception:
    from langchain.docstore.document import Document # fallback
from langchain_community.document_loaders import PyPDFLoader
import os
from datetime import datetime
from qdrant_client.models import VectorParams, Distance, PointStruct
import docx2txt
from pypdf import PdfReader
from langchain.chat_models import init_chat_model
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain.text_splitter import CharacterTextSplitter
try:
    from langchain_core.documents import Document # newer langchain
except Exception:
    from langchain.docstore.document import Document # fallback
from qdrant_client.models import VectorParams, Distance, PointStruct
from requests_toolbelt.multipart import decoder
import os, re , mimetypes, tempfile, urllib.parse
import pandas as pd
from typing import List, Optional
from pydantic import BaseModel, Field, field_validator
from langchain.output_parsers import PydanticOutputParser, OutputFixingParser
from langchain.prompts import PromptTemplate
from langchain.chat_models import ChatOpenAI
import json 
import numpy as np
import pandas as pd
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import CharacterTextSplitter  # ✅ new import
from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_qdrant import QdrantVectorStore


In [ ]:
def collections():
        r = requests.get("http://localhost:8000/all_collections", timeout=20)
        return r.json().get("collections", [])

collections()

In [ ]:
embeddings = OpenAIEmbeddings()

In [ ]:
def load_documents(folder_path: str) -> List[Document]:
    documents = []
    for filename in os.listdir(folder_path):
        file_path = os.path.join(folder_path, filename)
        if filename.endswith('.pdf'):
            loader = PyPDFLoader(file_path)
        else:
            print(f"Unsupported file type: {filename}")
            continue
        documents.extend(loader.load())
    return documents

folder_path = "pdf_reports/diageo/"

In [ ]:
reports = {}
documents = []
for path in os.listdir(folder_path):
    pth = folder_path + path
    docs = load_documents(pth)
    for doc in docs:
        doc.metadata["source"] = path
        documents.append(doc)


for doc in documents:
    key = doc.metadata["source"]
    if key not in reports.keys():
        reports[key]  = [doc]
    else:
        reports[key].append(doc)


In [ ]:
def collection(key: str, value: List[Document]):
    name = re.sub(r'[^a-zA-Z0-9_]+', '_', key)
    dim = len(embeddings.embed_query("dim?"))
    existing = [c.name for c in client_qd.get_collections().collections]
    if name not in existing:
        client_qd.create_collection(
        collection_name=name,
        vectors_config=VectorParams(size=dim, distance=Distance.COSINE),
    )
    
    qvs = QdrantVectorStore(
        client=client_qd,
        collection_name=name,
        embedding=embeddings,
    )

    qvs.add_documents(documents=value)


# client_qd = QdrantClient(path="./qdrant_data")
# qvs = QdrantVectorStore(client=client_qd, collection_name=name, embedding=embeddings)

In [ ]:
# from langchain_text_splitters import RecursiveCharacterTextSplitter
# for key, value in reports.items():
#     splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(chunk_size=1200, chunk_overlap=200)
#     chunks = splitter.split_documents(value)
#     collection(key, chunks)

In [ ]:
class Sub_name(BaseModel):
    sub_name: str = Field(description="Subtitle's name")
class Important_info(BaseModel):
    question: str = Field(description="Important information")
class Quantity_important_rows(BaseModel):
    question:int = Field(description="Quantity of important rows")
class Year(BaseModel):
    question: datetime = Field(description="Fiscal year")
class Period_start(BaseModel):  
    question: datetime = Field(description="Period start date")
class Period_end(BaseModel):
    question: datetime = Field(description="Period end date")
class Net_sales(BaseModel):
    question: float = Field(-1, description="Give me Net sales(absolute)  if you dont find it output should be -1")
class Revenue_growth_pct(BaseModel):            
    question: float = Field(-1, description="GIve me Revenue growth percentage if you dont find it output should be -1")
class Operating_profit(BaseModel):
    question: int = Field(-1, description="Give me Consolidated income statement as Operating profit if you dont find it output should be -1")
class Operating_margin(BaseModel):
    question: float = Field(-1, description="Give me Operating margin percentage if you dont find it output should be -1")
class Net_income_margin_pct(BaseModel):
    question: float = Field(-1, description="Find me Net income margin percentage if you don't find it output should be -1")
class Net_profit(BaseModel):
    question: int = Field(-1, description="Give me Net profit if you dont find it output should be -1")
class Eps(BaseModel):
    question: float = Field(-1, description="Give me EPS (basic/diluted) if you dont find it output should be -1")
class Cash_flow(BaseModel):
    question: int = Field(-1, description="Give me Free cash flow / Net cash from ops if you dont find it output should be -1")
class Capex(BaseModel):
    question: int = Field(-1, description="Give me Capex (absolute and/or cadence) if you dont find it output should be -1")
class Opex(BaseModel):
    question: int = Field(-1, description="Give me Opex, if you dont find output should be -1")
class Gross_profit(BaseModel):
    question: int = Field(-1, description="Give me Gross profit if you dont find it output should be -1")
class Share_of_sales(BaseModel):
    question: float = Field(description="Find me percentage of Share of sales by region, if you don't find it output should be -1")
class Gross_margin(BaseModel):
    question: float = Field(-1, description="Give me Gross margin percentage if you dont find it output should be -1")
class Revenue(BaseModel):
    question: int = Field(-1, description="Give me total revenue and clearly specify the currency (EUR, USD, etc.) and unit (e.g., millions, billions). otherwise output should be -1")
class Currency(BaseModel):
    question: str = Field("Unknown", description="Give me the Currency of this report (e.g., EUR, USD, GBP, etc.) otherwise output should be 'Unknown'")
class Revenue_unit(BaseModel):
    question: str = Field("Unknown", description="Specify the unit for all financial amounts, e.g., 'in millions of euros', 'in billions of USD' otherwise output should be 'Unknown'")
class Net_sales_unit(BaseModel):
    question: str = Field("Unknown", description="Specify the unit for Net Sales amount, e.g., 'in millions of euros', 'in billions of USD' otherwise output should be 'Unknown'")
class Operating_income(BaseModel):
    question: int = Field(-1, description="Give me Operating income (ROC) if you don't find it output should be -1")
class Operating_income_unit(BaseModel):
    question: str = Field("Unknown", description="Specify the unit for Operating Income amount, e.g., 'in millions of euros', 'in billions of USD' otherwise output should be 'Unknown'")
class Net_income(BaseModel):
    question: int = Field(-1, description="Give me Net income (Group share) if you don't find it output should be -1")
class Net_income_growth(BaseModel):
    question: float = Field(-1, description="Give me Net income growth percentage if you don't find it output should be -1")
class Net_debt(BaseModel):
    question: float = Field(-1, description="Give me Net debt if you don't find it output should be -1")
class Net_debt_to_ebitda(BaseModel):
    question: float = Field(-1, description="Give me Net debt/EBITDA ratio if you don't find it output should be -1")
class Dividend(BaseModel):
    question: float = Field(-1, description="Give me the proposed dividend per share (e.g., €4.70) if you don't find it output should be -1")
class Pro(BaseModel):
    question: float = Field(-1, description="Give me PRO (Profit from Recurring Operations) amount if you don't find it output should be -1")
class Pro_growth(BaseModel):
    question: float = Field(-1, description="Give me organic PRO growth percentage if you don't find it output should be -1")
class Free_cash_flow_amount(BaseModel):
    question: float = Field(-1, description="Free Cash Flow in €m")
class Free_cash_flow_unit(BaseModel):
    question: str = Field("Unknown", description="Specify the unit for Free Cash Flow amount, e.g., 'in millions of euros', 'in billions of USD' otherwise output should be 'Unknown'")
class Net_debt_change_amount(BaseModel):
    question: float = Field(-1, description="Change in Net Debt vs previous year in €m (use sign)")
class Net_debt_ending_amount(BaseModel):
    question: float = Field(-1, description="Net Debt at this year in €m")
class Net_debt_to_ebitda_ratio(BaseModel):
    question: float = Field(-1, description="Net Debt / EBITDA ratio at average rate")
class Dividend_per_share_proposed(BaseModel):
    question: float = Field(-1, description="Proposed dividend per share in €")
class Agm_date(BaseModel):
    question: str = Field('Unknown', description="AGM date for dividend approval (text) otherwise 'Unknown'")
class Dividend_cagr_since_fy19_pct(BaseModel):
    question: float = Field(-1, description="Dividend CAGR since FY19, %")
class Financial_policy_unchanged(BaseModel):
    question: bool = Field(True, description="Whether financial policy remains unchanged")
class Name(BaseModel):
    question: str = Field("Unknown", description="Give me Region of affiliates, if you don't find it output should be 'Unknown'")
class Net_sales(BaseModel):
    question: int = Field(-1, description="Find me Segmental Net sales  by region if you don't find it output should be -1")
class Revenue_region(BaseModel):
    question: int = Field(-1, description="Find me  Revenue by region if you don't find it output should be -1")
class Revenue_growth(BaseModel):
    question: int = Field(-1, description="Give me revenue growth by region if you don't find it output should be -1")
class Yoy_pct(BaseModel):
    question: int = Field(-1, description="Find me Region Year over Year percentages if you don't find it output should be -1")
class Currency_region(BaseModel):
    question: str = Field("Unknown", description="Currency symbol/code found for this region’s figures (e.g., '€', 'EUR', 'USD'). If not stated at region level, inherit from report. If unknown, return 'Unknown'.")
class Amount_unit(BaseModel):
    question: str = Field("Unknown", description="Exact scale stated for amounts for this region (e.g., 'en millions', 'en milliards', 'Md€', 'M€'). Always keep the unit’s wording as written. If not stated, return 'Unknown'.")
class Revenue_absolute(BaseModel):
    question: int = Field(-1, description="Absolute revenue for the region in the currency/scale above (extract digits only, no unit). If not given for the region, return -1.")
class Revenue_absolute_unit(BaseModel):
    question: str = Field("Unknown", description="Unit for the absolute revenue for the region (e.g., 'millions of euros', 'billions of USD'). If not found, return 'Unknown'.")
class Revenue_org_change_pct(BaseModel):
    question: float = Field(-1, description="Organic revenue change for the region as a signed percentage (e.g., -3.0 for -3.0%). If not found, return -1.")
class Revenue_pub_change_pct(BaseModel):
    question: float = Field(-1, description="Published (données publiées) revenue change for the region as a signed percentage. If not found, return -1.")
class Volume_growth_pct(BaseModel):
    question: float = Field(-1, description="Volume growth for the region as a signed percentage (e.g., +2.0) also if the text refers to Group level return -1.")
class Quantity_key_brands(BaseModel):
    question: int = Field(description="Find me Total Quantity of Key brands from this company")
class Key_brands(BaseModel):
    question: str = Field(description="Find me All Key brands from this company")
class Brand_companies(BaseModel):
    question: str = Field(description="Find me All Brand Companies from this company")
class Quantity_brand_companies(BaseModel):
    question: int = Field(-1, description="Find me Total Quantity of Brand Companies from this company otherwise output should be -1")
class Strategic_international_brands(BaseModel):
    question: str = Field(description="Find me All Strategic International Brands from this company")
class Quantity_strategic_international_brands(BaseModel):
    question: int = Field(-1, description="Find me Total Quantity of Strategic International Brands from this company otherwise output should be -1")
class Prestige_brands(BaseModel):
    question: str = Field(description="Find me All Prestige Brands from this company")
class Quantity_prestige_brands(BaseModel):
    question: int = Field(-1, description="Find me Total Quantity of Prestige Brands from this company otherwise output should be -1")
class Specialty_brands(BaseModel):
    question: str = Field(description="Find me All Specialty Brands from this company")
class Quantity_specialty_brands(BaseModel):
    question: int = Field(-1, description="Find me Total Quantity of Specialty Brands from this company otherwise output should be -1")
class Strategic_local_brands(BaseModel):
    question: str = Field(description="Find me All Strategic Local Brands from this company")
class Quantity_strategic_local_brands(BaseModel):
    question: int = Field(-1, description="Find me Total Quantity of Strategic Local Brands from this company otherwise output should be -1")
class Non_alcoholic_brands(BaseModel):
    question: str = Field(description="Find me All Non-Alcoholic Brands from this company")
class Quantity_non_alcoholic_brands(BaseModel):
    question: int = Field(-1, description="Find me Total Quantity of Non-Alcoholic Brands from this company otherwise output should be -1")
class Ready_to_drink_brands(BaseModel):
    question: str = Field(description="Find me All Ready-To-Drink Brands from this company")
class Quantity_ready_to_drink_brands(BaseModel):
    question: int = Field(-1, description="Find me Total Quantity of Ready-To-Drink Brands from this company otherwise output should be -1")
class Total_net_sales(BaseModel):
    question: float = Field(-1, description="Total FY Net Sales in million euros (absolute)")
class Organic_decline_pct(BaseModel):
    question: float = Field(-1, description="Organic decline percentage in total net sales")
class Reported_decline_pct(BaseModel):
    question: float = Field(-1, description="Reported decline percentage in total net sales")
class Net_sales_analysis_by_period(BaseModel):
    question: int = Field(description="Net Sales analysis by period and region ")
class Fx_impact(BaseModel):
    question: float = Field(-1, description="FX impact in million euros")
class Perimeter_impact(BaseModel):
    question: float = Field(-1, description="Positive perimeter impact in million euros")
class Americas_growth(BaseModel):
    question: float = Field(-1, description="Americas overall sales growth percentage")
class Usa_growth(BaseModel):
    question: float = Field(-1, description="USA sales growth percentage otherwise output should be -1")
class Asia_row_growth(BaseModel):
    question: float = Field(-1, description="Asia-Rest of World overall sales growth percentage otherwise output should be -1")
class China_growth(BaseModel):
    question: float = Field(-1, description="China sales growth percentage otherwise output should be -1")
class India_growth(BaseModel):
    question: float = Field(-1, description="India sales growth percentage otherwise output should be -1")
class Spirits_market_trend_value(BaseModel):
    question: float = Field(-1, description="Estimated global spirits market growth or normalization percentage otherwise -1")
class Spirits_market_trend(BaseModel):
    question: str = Field('Unknown', description="Trend summary for global spirits market normalization otherwise output should be 'Unknown'")
class Inventory_adjustments(BaseModel):
    question: str = Field('Unknown', description="Summary of inventory adjustments mentioned in report OTHERWISE output should be 'Unknown'")
class Pro_amount(BaseModel):
    question: float = Field(-1, description="FY24 Profit from Recurring Operations in €m otherwise output should be -1")
class Pro_organic_growth_pct(BaseModel):
    question: float = Field(-1, description="Organic growth percentage of PRO otherwise output should be -1")
class Pro_reported_change_pct(BaseModel):
    question: float = Field(-1, description="Reported change percentage of PRO otherwise output should be -1")
class Gross_margin_expansion_bps(BaseModel):
    question: int = Field(-1, description="Organic gross margin expansion in basis points otherwise output should be -1")
class Ap_amount(BaseModel):
    question: float = Field(-1, description="A&P spend in €bn otherwise output should be -1")
class Ap_pct_of_net_sales(BaseModel):
    question: float = Field(-1, description="A&P as percentage of Net Sales otherwise output should be -1")
class Operating_margin_org_bps(BaseModel):
    question: int = Field(-1, description="Operating margin organic expansion in basis points otherwise output should be -1")
class Operating_margin_org_pct(BaseModel):
    question: float = Field(-1, description="Operating margin on an organic basis, %")
class Operating_margin_reported_pct(BaseModel):
    question: float = Field(-1, description="Operating margin on a reported basis, %")
class Fx_impact_amount(BaseModel):
    question: float = Field(-1, description="Adverse FX impact on reported operating margin in €m")
class Perimeter_effect_amount(BaseModel):
    question: float = Field(-1, description="Favourable perimeter effects in €m")
class Group_share_net_pro_amount(BaseModel):
    question: float = Field(-1, description="Group share of Net PRO in €m")
class Group_share_net_pro_amount_unit(BaseModel):
    question: str = Field("Unknown", description="Specify the unit for Group Share of Net PRO amount, e.g., 'in millions of euros', 'in billions of USD' otherwise output should be 'Unknown'")
class Group_share_net_pro_change_pct(BaseModel):
    question: float = Field(-1, description="YoY change percentage of Group share of Net PRO")
class Avg_cost_of_debt_pct(BaseModel):
    question: float = Field(-1, description="Average cost of debt percentage for recurring financial expenses")
class Group_share_net_profit_amount(BaseModel):
    question: float = Field(-1, description="Group Share of Net Profit in €m")
class Group_share_net_profit_change_pct(BaseModel):
    question: float = Field(-1, description="YoY change percentage of Group Share of Net Profit")
class Eps_amount(BaseModel):
    question: float = Field(-1, description="Earnings per share in €")
class Headquarters(BaseModel):
    question: str = Field(description="Give me Headquarters location of this company")
class Executive_committee_examples(BaseModel):
    question: str = Field('Unknown', description="Give me all of executive committee members")
class Executive_committee_quantity(BaseModel):
    question: int = Field(description="Give me Total quantity of executive committee members")
class Board_of_directors_examples(BaseModel):
    question: str = Field('Unknown', description="Give me all of board of directors members otherwise output should be 'Unknown'")
class Board_of_directors_quantity(BaseModel):
    question: int = Field(description="Give me Total quantity of board of directors members")
class Affiliate_name(BaseModel):
    question: str = Field("Unknown", description="Exact all names of the affiliate or subsidiary company (e.g., 'Pernod Ricard USA', 'Pernod Ricard India'). If not mentioned in the report, return 'Unknown'.")
class Affiliates(BaseModel):
    question: str = Field('Unknown', description="Give me all of names of affiliates of this company or subsidiary company (e.g., 'Pernod Ricard USA', 'Pernod Ricard India').otherwise output should be 'Unknown'")
class Affiliate_quantity(BaseModel):
    question: int = Field(description="Give me Total quantity of affiliates of this company or subsidiary companyof this company")
class Total_employees(BaseModel):
    question: int = Field(-1, description="Give me Total employees quantity")
class Avg_age(BaseModel):
    question: int = Field(-1, description="Give me Average age of employees")
class Qty_nationalities(BaseModel):
    question: int = Field(-1, description="Find me total Quantity of Nationalities if you dont find it output should be -1")
class Pct_women(BaseModel):
    question: int = Field(-1, description="Give percentage of women in company otherwise output should be -1")
class Leadership_pro(BaseModel):
    question: int = Field(-1, description="Give me percentage of leadership programs otherwise output should be -1")
class Senior_appointments_bw(BaseModel):
    question: int = Field(-1, description="Give me percentage of Senior appointments by women otherwise output should be -1")
class Female_leaders(BaseModel):
    question: int = Field(-1, description="Give me percentage of female leaders otherwise output should be -1")
class Ethn_diverse_lead(BaseModel):
    question: int = Field(-1, description="Give me percentage of ethnically diverse leaders otherwise output should be -1")
class Employees_with_disabilities_pct(BaseModel):
    question: int = Field(-1, description="Give me percentage of employees with disabilities otherwise output should be -1")
class Global_mobility_pct(BaseModel):
    question: int = Field(-1, description="Give me percentage of global mobility otherwise output should be -1")
class Nationalities(BaseModel):
    question: int = Field(-1, description="Give me quantity of nationalities otherwise output should be -1")
class Total_employees_social(BaseModel):
    question: int = Field(-1, description="Total number of employees")
class Women_in_workforce_pct(BaseModel):
    question: float = Field(-1, description="Percentage of women in total workforce, otherwise -1")
class Women_in_management_pct(BaseModel):
    question: float = Field(-1, description="Percentage of women in management roles, otherwise -1")
class Ethnically_diverse_leaders_pct(BaseModel):
    question: float = Field(-1, description="Percentage of ethnically diverse leaders, otherwise -1")
class Pay_gap_ratio(BaseModel):
    question: float = Field(-1, description="Gender or diversity pay gap ratio (men vs women), otherwise -1")
class Employees_with_disabilities_pct_social(BaseModel):
    question: float = Field(-1, description="Percentage of employees with disabilities, otherwise -1")
class Lgbtq_inclusion_programs(BaseModel):
    question: str = Field('Unknown', description="Information about LGBTQ+ inclusion or diversity initiatives otherwise output should be 'Unknown'")
class Training_hours_per_employee(BaseModel):
    question: float = Field(-1, description="Average training hours per employee, otherwise -1")
class Community_investment_amount(BaseModel):
    question: float = Field(-1, description="Community investments or donations in €m, otherwise -1")
class Carbon_emissions_total(BaseModel):
    question: float = Field(-1, description="Total carbon emissions (Scope 1+2) in metric tons CO2e, otherwise -1")
class Carbon_emissions_scope3(BaseModel):
    question: float = Field(-1, description="Scope 3 emissions in metric tons CO2e, otherwise -1")
class Emission_intensity(BaseModel):
    question: float = Field(-1, description="Emission intensity (tons CO2e per unit of revenue), otherwise -1")
class Renewable_energy_pct(BaseModel):
    question: float = Field(-1, description="Percentage of renewable energy used, otherwise -1")
class Energy_consumption_total(BaseModel):
    question: float = Field(-1, description="Total energy consumption (MWh), otherwise -1")
class Water_withdrawal_total(BaseModel):
    question: float = Field(-1, description="Total water withdrawal in cubic meters, otherwise -1")
class Waste_recycled_pct(BaseModel):
    question: float = Field(-1, description="Percentage of total waste recycled, otherwise -1")
class Biodiversity_initiatives(BaseModel):
    question: str = Field('Unknown', description="Summary of biodiversity protection or environmental initiatives otherwise output should be 'Unknown'")
class Water_efficiency(BaseModel):
    question: str = Field('Unknown', description="Find me information about Water efficiency otherwise output should be 'Unknown'")
class Energy_consumption(BaseModel):
    question: str = Field('Unknown', description="Find me information about Energy consumption otherwise output should be 'Unknown'")
class Distillery_water(BaseModel):
    question: str = Field('Unknown', description="Find me information about Distillery water otherwise output should be 'Unknown'")
class Responsible_consumption(BaseModel):
    question: str = Field('Unknown', description="Find me information about Responsible consumption otherwise output should be 'Unknown'")

In [ ]:
# class Articles(BaseModel):
#     topic: str = Field(description="Find me main article of this report")
#     company_name: str = Field(description="Give me Company's name")
#     motto: str = Field(description="Give me Company's motto or slogan")
#     founded_group: str = Field(description="Give me Founded year of this company")
#     headquarters: str = Field(description="Give me Headquarters location of this company i need city and country")
#     maisons_count: int = Field(description="Give me Total quantity of maisons of this company")
#     countries: int = Field(description="Give me Total quantity of countries where this company is present")
#     stores_count: int = Field(description="Give me Total quantity of stores of this company")
#     fy: List[FiscalYear] = Field(description="Fiscal Years")
#     region: List[Region] = Field(description="Regions KPTs")
#     finance: List[Financials] = Field(description="Financial and capital KPTs")
#     free_cash_flow_debt: List[FreeCashFlow_Debt] = Field(description="Free Cash Flow and Debt KPTs")
#     sales_drinks: List[Sales_Drinks] = Field(description="Sales Drinks KPTs")
#     results_drinks: List[Results_Drinks] = Field(description="Results Drinks KPTs")
#     corporate: List[Corporate_information] = Field(description="Corporate information KPTs")
#     social_dei: List[Social_DEI] = Field(description="Social and DEI KPTs")
#     environmental: List[Environmental] = Field(description="Environmental KPTs")
#     governance: List[Governance] = Field(description="Governance KPTs")
#     subtitles: List[Subtitle] = Field(description="subtitles")

#### The code loads PDF documents from a specified folder, processes their content using a language model to extract structured data based on predefined schemas, and saves the extracted information as JSON files.

In [ ]:
# from langchain_text_splitters import RecursiveCharacterTextSplitter
# for key, value in reports.items():
#     splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(chunk_size=250000, chunk_overlap=0)
#     chunks = splitter.split_text(value)
#     print(f"Chunks for {key}: {chunks}")

In [ ]:
group_fields = {
    "FiscalYear": [Year , Period_start , Period_end],
    "Region": [Name , Net_sales , Revenue_region , Revenue_growth , Yoy_pct , Currency_region , Amount_unit , Revenue_absolute , Revenue_absolute_unit , Revenue_org_change_pct , Revenue_pub_change_pct , Volume_growth_pct ],
    "Financials": [Net_sales , Revenue_growth_pct , Operating_profit , Operating_margin , Net_income_margin_pct , Net_profit , Eps , Cash_flow , Capex , Opex , Gross_profit , Share_of_sales , Gross_margin , Revenue , Currency , Revenue_unit , Net_sales_unit , Operating_income , Operating_income_unit , Net_income , Net_income_growth , Net_debt , Net_debt_to_ebitda , Dividend , Pro , Pro_growth ],
    "FreeCashFlow_Debt": [Free_cash_flow_amount,Free_cash_flow_unit,Net_debt_change_amount,Net_debt_ending_amount,Net_debt_to_ebitda_ratio,Dividend_per_share_proposed,Agm_date,Dividend_cagr_since_fy19_pct,Financial_policy_unchanged],
    "Brands": [Quantity_key_brands, Key_brands, Brand_companies, Quantity_brand_companies, Strategic_international_brands, Quantity_strategic_international_brands, Prestige_brands, Quantity_prestige_brands, Specialty_brands, Quantity_specialty_brands, Strategic_local_brands, Quantity_strategic_local_brands, Non_alcoholic_brands, Quantity_non_alcoholic_brands, Ready_to_drink_brands, Quantity_ready_to_drink_brands],
    "Sales_Drinks": [Total_net_sales,Organic_decline_pct,Reported_decline_pct,Net_sales_analysis_by_period,Fx_impact,Perimeter_impact,Americas_growth,Usa_growth,Asia_row_growth,China_growth,India_growth,Spirits_market_trend_value,Spirits_market_trend,Inventory_adjustments],
    "Results_Drinks": [Pro_amount,Pro_organic_growth_pct,Pro_reported_change_pct,Gross_margin_expansion_bps,Ap_amount,Ap_pct_of_net_sales,Operating_margin_org_bps,Operating_margin_org_pct,Operating_margin_reported_pct,Fx_impact_amount,Perimeter_effect_amount,Group_share_net_pro_amount,Group_share_net_pro_amount_unit,Group_share_net_pro_change_pct,Avg_cost_of_debt_pct,Group_share_net_profit_amount,Group_share_net_profit_change_pct, Eps_amount],
    "Corporate_information": [Headquarters,Executive_committee_examples,Executive_committee_quantity,Board_of_directors_examples,Board_of_directors_quantity,Affiliate_name,Affiliates,Affiliate_quantity,Total_employees,Avg_age,Qty_nationalities],
    "Social_DEI": [Total_employees_social, Women_in_workforce_pct, Women_in_management_pct, Ethnically_diverse_leaders_pct, Pay_gap_ratio, Employees_with_disabilities_pct_social, Lgbtq_inclusion_programs, Training_hours_per_employee, Community_investment_amount],
    "Environmental": [Carbon_emissions_total, Carbon_emissions_scope3, Emission_intensity, Renewable_energy_pct, Energy_consumption_total, Water_withdrawal_total, Waste_recycled_pct, Biodiversity_initiatives],
    "Governance": [Water_efficiency, Energy_consumption, Distillery_water, Responsible_consumption],
    "Subtitle":  [Sub_name, Important_info, Quantity_important_rows],
}

In [ ]:
import io
from fastapi.responses import StreamingResponse

llm = init_chat_model("openai:gpt-5", temperature=1)
dim = len(embeddings.embed_query("dim?"))


def ensure_collection(name: str, dim: int):
    existing = [c.name for c in client_qd.get_collections().collections]
    if name not in existing:
        client_qd.create_collection(
            collection_name=name,
            vectors_config=VectorParams(size=dim, distance=Distance.COSINE),
        )

splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(chunk_size=1200, chunk_overlap=200)
def text_into_qdrant(collection_name: str, text: str):
    docs = [Document(page_content=chunk) for chunk in splitter.split_text(text)]
    qvs = QdrantVectorStore(client=client_qd, collection_name=collection_name, embedding=embeddings)
    if docs:
        qvs.add_documents(docs)

def prompt_question(qvs, model_cls):
    question = model_cls.model_fields['question'].description
    hits = qvs.similarity_search(question, k=3)
    context = "\n".join(doc.page_content for doc in hits)
    parser = PydanticOutputParser(pydantic_object=model_cls)
    parser = OutputFixingParser.from_llm(parser=parser, llm=llm)

    prompt = PromptTemplate.from_template(
        "i received a company file and i need to extract data : {abc} "
        "Use ONLY the context to answer. If a value is not found, use -1 or 'Unknown' per the schema.\n"
        "question: {question}\ncontext: {text}"
    )
    chain = prompt | llm | parser
    parsed = chain.invoke({
        "abc": parser.get_format_instructions(),
        "question": question,
        "text": context
    })
    return parsed["question"] if isinstance(parsed, dict) else getattr(parsed, "question", parsed)


@app.get("/all_collections")
async def all_collections():
    existing = [c.name for c in client_qd.get_collections().collections]
    all_frames: Dict[str, pd.DataFrame] = {}
    for sheet_name, class_list in group_fields.items():
        cols = [cls.__name__ for cls in class_list]
        rows = []
        for coll in existing:
            qvs = QdrantVectorStore(client=client_qd, collection_name=coll, embedding=embeddings)
            row = []
            for model_cls in class_list:
                value = prompt_question(qvs, model_cls)
                row.append(value)
            rows.append(row)
            
        df = pd.DataFrame(rows, columns=cols, index=existing)
        all_frames[sheet_name] = df

    buf = io.BytesIO()
    with pd.ExcelWriter(buf, engine="openpyxl") as w:
        for sheet, df in all_frames.items():
            wsheet = sheet[:31]
            df.to_excel(w, sheet_name=wsheet)
    buf.seek(0)
    return StreamingResponse(
        io.BytesIO(buf.read()),
        media_type="application/vnd.openxmlformats-officedocument.spreadsheetml.sheet",
        headers={"Content-Disposition": 'attachment; filename="report.xlsx"'},
    )









In [ ]:
existing = [c.name for c in client_qd.get_collections().collections]

all_dfs = {}

for key , cls in group_fields.items():
    llm = init_chat_model("openai:gpt-5", temperature=1)
    arr = []
    for c in cls:
        parser = PydanticOutputParser(pydantic_object=c)
        parser = OutputFixingParser.from_llm(parser=parser, llm=llm)
        question = c.model_fields['question'].description
        chunks = []
        col = []
        for name in existing: 
            qvs = QdrantVectorStore(client=client_qd, collection_name=name, embedding=embeddings)
            answ = qvs.similarity_search(question, k=2)
            chunks += [doc.page_content for doc in answ]
            context = "\n".join(chunks)

            prompt = PromptTemplate.from_template("""
            i recieved a company file and i need to extract data : {abc} Use ONLY the context to answer. If a value is not found, use -1 or 'Unknown' per the schema.
                question : {question} , context : {text} """.strip())
            chain = prompt | llm | parser 
            result = chain.invoke({
            "abc": parser.get_format_instructions(),
            "question": question,
            "text": context
            }).model_dump()
            col.append(result["question"])
        
        arr.append(col)
    df = pd.DataFrame(np.array(arr).T, columns=[c.__name__ for c in cls])
    df.index = existing

    all_dfs[key] = df

    buf = io.BytesIO()
    with pd.ExcelWriter(buf, engine="openpyxl") as w:
        for sheet, df in all_dfs.items():
            wsheet = sheet[:31]
            df.to_excel(w, sheet_name=wsheet)
    buf.seek(0)
    return StreamingResponse(
        io.BytesIO(buf.read()),
        media_type="application/vnd.openxmlformats-officedocument.spreadsheetml.sheet",
        headers={"Content-Disposition": 'attachment; filename="report.xlsx"'},
    )
    break

In [ ]:
all_dfs

In [ ]:
def collections():
        r = requests.get("http://localhost:8000/all_collections", timeout=20)
        return r.json().get("collections", [])

In [ ]:
def question_to_chunks(question: str , k: int =10):
    existing = [c.name for c in client_qd.get_collections().collections] 
    chunks = []
    for name in existing: 
        qvs = QdrantVectorStore(client=client_qd, collection_name=name, embedding=embeddings)
        for q in list_q:
            answ = qvs.similarity_search(q, k=5)
            print(answ)
            chunks += [doc.page_content for doc in answ]
            print(chunks)
    #return chunks[:k]
question_to_chunks("")

In [ ]:
def prompt_llm(question: str , context: str)-> dict:
    llm = init_chat_model("openai:gpt-5", temperature=1)
    parser = PydanticOutputParser(pydantic_object=)
    parser = OutputFixingParser.from_llm(parser=parser, llm=llm)
    prompt = PromptTemplate.from_template("""
        i recieved a company file and i need to extract data : {abc} Use ONLY the context to answer. If a value is not found, use -1 or 'Unknown' per the schema.
         question : {question} , context : {text} """.strip())
    chain = prompt | llm | parser 
    result = chain.invoke({
    "abc": parser.get_format_instructions(),
    "question": question,
    "text": context
    })
    return result.model_dump()


for question in questions:
    context = question_to_chunks(question)
    result = prompt_llm(question, context)
    splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(chunk_size=1500, chunk_overlap=200)
    chunks = splitter.split_text(result)
    
    for chunk in chunks:
        result = prompt_llm(chunk)
        clean_file_name = key.split("\\")[-1].replace(".pdf", "").replace("pdf_reports/", "")
        with  open(f"jsons/{clean_file_name}.json", "w") as f:
            json.dump(result, f, indent=4)

In [ ]:
def prompt_for_llm(text: str)-> dict:
    llm = init_chat_model("openai:gpt-5", temperature=1)
    parser = PydanticOutputParser(pydantic_object=Articles)
    parser = OutputFixingParser.from_llm(parser=parser, llm=llm)
    prompt = PromptTemplate.from_template("""
        i recieved a company file and i need to extract data : {abc} from  the next text : {text} 
        """)
    chain = prompt | llm | parser 
    result = chain.invoke({
    "abc": parser.get_format_instructions(),
    "text": text
    })
    return result.model_dump()


for key, value in reports.items():
    splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(chunk_size=250000, chunk_overlap=0)
    chunks = splitter.split_text(value)
    for chunk in chunks:
        result = prompt_for_llm(chunk)
        clean_file_name = key.split("\\")[-1].replace(".pdf", "").replace("pdf_reports/", "")
        with  open(f"jsons/{clean_file_name}.json", "w") as f:
            json.dump(result, f, indent=4)


In [ ]:
# def prompt_for_llm(text: str)-> dict:
#     llm = init_chat_model("openai:gpt-5", temperature=1)
#     parser = PydanticOutputParser(pydantic_object=Articles)
#     parser = OutputFixingParser.from_llm(parser=parser, llm=llm)
#     prompt = PromptTemplate.from_template("""
#         i recieved a company file and i need to extract data : {abc} from  the next text : {text} 
#         """)
#     chain = prompt | llm | parser 
#     result = chain.invoke({
#     "abc": parser.get_format_instructions(),
#     "text": text
#     })
#     return result.model_dump()


# for source, text in reports.items():
#     result = prompt_for_llm(text)
#     clean_file_name = source.split("\\")[-1].replace(".pdf", "").replace("pdf_reports/", "")
#     with  open(f"jsons/{clean_file_name}.json", "w") as f:
#         json.dump(result, f, indent=4)

## Diageo tables

In [ ]:
with open("jsons/diago/annual-report-2025_diageo.json") as f:
    di25 = json.load(f)
fy_25 = pd.DataFrame(di25["fy"])
fy_25["company name"] = "25-Diageo"
region25 = pd.DataFrame(di25["region"])
region25["company name"] = "25-Diageo"
finance25 = pd.DataFrame(di25["finance"])
finance25["company name"] = "25-Diageo"
free_cash_flow_debt_25 = pd.DataFrame(di25["free_cash_flow_debt"])
free_cash_flow_debt_25["company name"] = "25-Diageo"
results_drinks_25 = pd.DataFrame(di25["results_drinks"])
results_drinks_25["company name"] = "25-Diageo"
corporate_information_25 = pd.DataFrame(di25["corporate"])
corporate_information_25["company name"] = "25-Diageo"
social_dei_25 = pd.DataFrame(di25["social_dei"])
social_dei_25["company name"] = "25-Diageo"

with open("jsons/diago/financial-statements.json") as f:
    di25v = json.load(f)
sales_drinks_25 = pd.DataFrame(di25v["sales_drinks"])
sales_drinks_25["company name"] = "25-Diageo"   
di25

In [ ]:
with open("jsons/diago/f24-Diageo.json") as f:
    di24 = json.load(f)
fy_24 = pd.DataFrame(di24["fy"])
fy_24["company name"] = "Diageo_24"
region = pd.DataFrame(di24["region"])
region["company name"] = "Diageo_24"
finance_full = pd.DataFrame(di24["finance"])
finance_full["company name"] = "Diageo_24"
free_cash_flow_debt_24 = pd.DataFrame(di24["free_cash_flow_debt"])
free_cash_flow_debt_24["company name"] = "Diageo_24"
sales_drinks_24 = pd.DataFrame(di24["sales_drinks"])
sales_drinks_24["company name"] = "Diageo_24"
results_drinks = pd.DataFrame(di24["results_drinks"])
results_drinks["company name"] = "Diageo_24"
corporate_information = pd.DataFrame(di24["corporate"])
corporate_information["company name"] = "Diageo_24"
social_dei = pd.DataFrame(di24["social_dei"])
social_dei["company name"] = "Diageo_24"


In [ ]:
with open("jsons/diago/f24-Diageo75.json") as f:
    di24 = json.load(f)
finance = pd.DataFrame(di24["finance"])
finance["company name"] = "Diageo_24"
free_cash_flow_debtdeb = pd.DataFrame(di24["free_cash_flow_debt"])
free_cash_flow_debtdeb["company name"] = "Diageo_24"
sales_drinks_deb = pd.DataFrame(di24["sales_drinks"])
sales_drinks_deb["company name"] = "Diageo_24"
results_drinks_deb = pd.DataFrame(di24["results_drinks"])
results_drinks_deb["company name"] = "Diageo_24"
corporate_information24 = pd.DataFrame(di24["corporate"])
corporate_information24["company name"] = "Diageo_24"
social_dei24 = pd.DataFrame(di24["social_dei"])
social_dei24["company name"] = "Diageo_24"


In [ ]:
with open("jsons/diago/f24-Diageofin.json") as f:
    di24 = json.load(f)
finance_fin = pd.DataFrame(di24["finance"])
finance_fin["company name"] = "Diageo_24"
free_cash_flow_debt_fin = pd.DataFrame(di24["free_cash_flow_debt"])
free_cash_flow_debt_fin["company name"] = "Diageo_24"
sales_drinks_fin = pd.DataFrame(di24["sales_drinks"])
sales_drinks_fin["company name"] = "Diageo_24"
results_drinks_fin = pd.DataFrame(di24["results_drinks"])
results_drinks_fin["company name"] = "Diageo_24"

In [ ]:
with open("jsons/diago/diageo-annual-report-2023_deb.json") as f:
    di23_deb = json.load(f)
fy_23 = pd.DataFrame(di23_deb["fy"])
fy_23["company name"] = "Diageo_23"
region = pd.DataFrame(di23_deb["region"])
region["company name"] = "Diageo_23"
finance_fin = pd.DataFrame(di23_deb["finance"])
finance_fin["company name"] = "Diageo_23"
free_cash_flow_debt_deb = pd.DataFrame(di23_deb["free_cash_flow_debt"])
free_cash_flow_debt_deb["company name"] = "Diageo_23"
sales_drinks_23 = pd.DataFrame(di23_deb["sales_drinks"])
sales_drinks_23["company name"] = "Diageo_23"
results_drinks_23 = pd.DataFrame(di23_deb["results_drinks"])
results_drinks_23["company name"] = "Diageo_23"
corporate_information23 = pd.DataFrame(di23_deb["corporate"])
corporate_information23["company name"] = "Diageo_23"
social_dei = pd.DataFrame(di23_deb["social_dei"])
social_dei["company name"] = "Diageo_23"

In [ ]:
with open("jsons/diago/diageo-annual-report-2023_fin.json") as f:
    di23_fin = json.load(f)
region_fin = pd.DataFrame(di23_fin["region"])
region_fin["company name"] = "Diageo_23"
finance_fin = pd.DataFrame(di23_fin["finance"])
finance_fin["company name"] = "Diageo_23"
free_cash_flow_debt_fin = pd.DataFrame(di23_fin["free_cash_flow_debt"])
free_cash_flow_debt_fin["company name"] = "Diageo_23"
results_drinks = pd.DataFrame(di23_fin["results_drinks"])
results_drinks["company name"] = "Diageo_23"


In [ ]:
with open("jsons/diago/Annual report 2022 – Diageo_deb.json") as f:
    pr22 = json.load(f)
fy_22 = pd.DataFrame(pr22["fy"])
fy_22["company name"] = "22-Diageo"
region22 = pd.DataFrame(pr22["region"])
region22["company name"] = "22-Diageo"
finance_deb = pd.DataFrame(pr22["finance"])
finance_deb["company name"] = "22-Diageo"
free_cash_flow_debt_22 = pd.DataFrame(pr22["free_cash_flow_debt"])
free_cash_flow_debt_22["company name"] = "22-Diageo"
sales_drinks_22 = pd.DataFrame(pr22["sales_drinks"])
sales_drinks_22["company name"] = "22-Diageo"
results_drinks_22 = pd.DataFrame(pr22["results_drinks"])
results_drinks_22["company name"] = "22-Diageo"
corporate_information_22 = pd.DataFrame(pr22["corporate"])
corporate_information_22["company name"] = "22-Diageo"
social_dei22 = pd.DataFrame(pr22["social_dei"])
social_dei22["company name"] = "22-Diageo"


In [ ]:
with open("jsons/diago/Annual report 2022 – Diageo_fin.json") as f:
    pr22 = json.load(f)
finance_fin = pd.DataFrame(pr22["finance"])
finance_fin["company name"] = "22-Diageo"
results_drinks = pd.DataFrame(pr22["results_drinks"])
results_drinks["company name"] = "22-Diageo"

In [ ]:
with open("jsons/diago/254_annual-report-2020_diageo.json") as f:
    di25 = json.load(f)
results_drinks_20 = pd.DataFrame(di25["results_drinks"])
results_drinks_20["company name"] = "20-Diageo"
corporate_information_20 = pd.DataFrame(di25["corporate"])
corporate_information_20["company name"] = "20-Diageo"
social_dei20 = pd.DataFrame(di25["social_dei"])
social_dei20["company name"] = "20-Diageo"

In [ ]:
df = corporate_information.copy()

merged = {}
for col in df.columns:
    vals = (
        df[col]
        .replace([-1, -1.0, "-1"], np.nan)
        .dropna()
        .unique()
        .tolist()
    )
    merged[col] = vals if len(vals) > 0 else [np.nan]

corporate_information = pd.DataFrame([merged])
corporate_information = corporate_information.map(
    lambda x: ", ".join(map(str, x)) if isinstance(x, list) else x
)

In [ ]:
corporate_information = pd.concat([corporate_information,corporate_information24], ignore_index=True)

In [ ]:
result_drink_24 = pd.concat([results_drinks, results_drinks_deb, results_drinks_fin], ignore_index=True)

In [ ]:
df = result_drink_24.copy()

merged = {}
for col in df.columns:
    vals = (
        df[col]
        .replace([-1, -1.0, "-1"], np.nan)
        .dropna()
        .unique()
        .tolist()
    )
    merged[col] = vals if len(vals) > 0 else [np.nan]

result_drink_24 = pd.DataFrame([merged])
result_drink_24 = result_drink_24.map(
    lambda x: ", ".join(map(str, x)) if isinstance(x, list) else x
)

In [ ]:
sales_drink_24 = pd.concat([sales_drinks_24, sales_drinks_deb, sales_drinks_fin], ignore_index=True)

In [ ]:
df = sales_drink_24.copy()

merged = {}
for col in df.columns:
    vals = (
        df[col]
        .replace([-1, -1.0, "-1"], np.nan)
        .dropna()
        .unique()
        .tolist()
    )
    merged[col] = vals if len(vals) > 0 else [np.nan]

sales_drink = pd.DataFrame([merged])
sales_drink = sales_drink.map(
    lambda x: ", ".join(map(str, x)) if isinstance(x, list) else x
)

In [ ]:
finance_diageo_24 = pd.concat([finance_full, finance, finance_fin], ignore_index=True)

In [ ]:
df = finance_diageo_24.copy()

merged = {}
for col in df.columns:
    vals = (
        df[col]
        .replace([-1, -1.0, "-1"], np.nan)
        .dropna()
        .unique()
        .tolist()
    )
    merged[col] = vals if len(vals) > 0 else [np.nan]

finance_merged = pd.DataFrame([merged])
finance_merged = finance_merged.map(
    lambda x: ", ".join(map(str, x)) if isinstance(x, list) else x
)

In [ ]:
fcf_23 = pd.concat([free_cash_flow_debt_deb, free_cash_flow_debt_fin], ignore_index=True)

In [ ]:
df = fcf_23.copy()

merged = {}
for col in df.columns:
    vals = (
        df[col]
        .replace([-1, -1.0, "-1"], np.nan)
        .dropna()
        .unique()
        .tolist()
    )
    merged[col] = vals if len(vals) > 0 else [np.nan]

fcf_23 = pd.DataFrame([merged])
fcf_23 = fcf_23.map(
    lambda x: ", ".join(map(str, x)) if isinstance(x, list) else x
)

display(fcf_23)  

In [ ]:
region_23 = pd.concat([region, region_fin], ignore_index=True)

In [ ]:
finance_23 = pd.concat([finance_fin, finance_fin], ignore_index=True)

In [ ]:
df = finance_23.copy()

merged = {}
for col in df.columns:
    vals = (
        df[col]
        .replace([-1, -1.0, "-1"], np.nan)
        .dropna()
        .unique()
        .tolist()
    )
    merged[col] = vals if len(vals) > 0 else [np.nan]

finance = pd.DataFrame([merged])
finance = finance.map(
    lambda x: ", ".join(map(str, x)) if isinstance(x, list) else x
)

diageo_finance = pd.concat([finance_merged, finance], ignore_index=True)


In [ ]:
finance = pd.concat([finance_deb, finance_fin], ignore_index=True)

df = finance.copy()

merged = {}
for col in df.columns:
    vals = (
        df[col]
        .replace([-1, -1.0, "-1"], np.nan)
        .dropna()
        .unique()
        .tolist()
    )
    merged[col] = vals if len(vals) > 0 else [np.nan]

finance = pd.DataFrame([merged])
finance = finance.map(
    lambda x: ", ".join(map(str, x)) if isinstance(x, list) else x
)
diageo_finance = pd.concat([diageo_finance, finance], ignore_index=True)

In [ ]:
result_drinks_diageo = pd.concat([results_drinks_20, results_drinks_22 , results_drinks_23 ,result_drink_24 , results_drinks_25], ignore_index=True)
sales_drink_diageo = pd.concat([sales_drinks_22, sales_drinks_23 , sales_drink, sales_drinks_25], ignore_index=True)
free_cash_flow_diageo = pd.concat([free_cash_flow_debt_22, fcf_23 ,free_cash_flow_debt_24 , free_cash_flow_debt_25], ignore_index=True)
regions_diageo = pd.concat([region22, region_23, region, region25], ignore_index=True)
fy_diageo = pd.concat([fy_24, fy_23, fy_22, fy_25], ignore_index=True)
finances_diageo = pd.concat([diageo_finance, finance25], ignore_index=True)
corporate_information_diageo = pd.concat([ corporate_information_20,corporate_information_22, corporate_information23, corporate_information, corporate_information_25], ignore_index=True)
social_dei_diageo = pd.concat([social_dei20, social_dei22, social_dei , social_dei24 , social_dei_25], ignore_index=True)


## Pernod Ricard

In [ ]:
with open("jsons/pr/FR_FY20.json") as f:
    pr20 = json.load(f)
region20 = pd.DataFrame(pr20["region"])
region20["company name"] = "20-PernodRicard"
finance = pd.DataFrame(pr20["finance"])
finance["company name"] = "20-PernodRicard"
free_cash_flow_debt20 = pd.DataFrame(pr20["free_cash_flow_debt"])
free_cash_flow_debt20["company name"] = "20-PernodRicard"
sales_drinks20 = pd.DataFrame(pr20["sales_drinks"])
sales_drinks20["company name"] = "20-PernodRicard"
results_drinks20 = pd.DataFrame(pr20["results_drinks"])
results_drinks20["company name"] = "20-PernodRicard"
corporate_information20 = pd.DataFrame(pr20["corporate"])
corporate_information20["company name"] = "20-PernodRicard"
social_dei20 = pd.DataFrame(pr20["social_dei"])
social_dei20["company name"] = "20-PernodRicard"


In [ ]:
with open("jsons/pr/PR _FY21.json") as f:
    pr21 = json.load(f)
region21 = pd.DataFrame(pr21["region"])
region21["company name"] = "21-PernodRicard"
finance21 = pd.DataFrame(pr21["finance"])
finance21["company name"] = "21-PernodRicard"
free_cash_flow_debt21 = pd.DataFrame(pr21["free_cash_flow_debt"])
free_cash_flow_debt21["company name"] = "21-PernodRicard"
sales_drinks21 = pd.DataFrame(pr21["sales_drinks"])
sales_drinks21["company name"] = "21-PernodRicard"
results_drinks21 = pd.DataFrame(pr21["results_drinks"])
results_drinks21["company name"] = "21-PernodRicard"
corporate_information21 = pd.DataFrame(pr21["corporate"])
corporate_information21["company name"] = "21-PernodRicard"
social_dei21 = pd.DataFrame(pr21["social_dei"])
social_dei21["company name"] = "21-PernodRicard"

In [ ]:
with open("jsons/pr/Pernod_prFY24.json") as f:
    pr24 = json.load(f)
region24 = pd.DataFrame(pr24["region"])
region24["company name"] = "24-PernodRicard"
finance24 = pd.DataFrame(pr24["finance"])
finance24["company name"] = "24-PernodRicard"
free_cash_flow_debt24 = pd.DataFrame(pr24["free_cash_flow_debt"])
free_cash_flow_debt24["company name"] = "24-PernodRicard"
sales_drinks24 = pd.DataFrame(pr24["sales_drinks"])
sales_drinks24["company name"] = "24-PernodRicard"
results_drinks24 = pd.DataFrame(pr24["results_drinks"])
results_drinks24["company name"] = "24-PernodRicard"
corporate_information24 = pd.DataFrame(pr24["corporate"])
corporate_information24["company name"] = "24-PernodRicard"
social_dei24 = pd.DataFrame(pr24["social_dei"])
social_dei24["company name"] = "24-PernodRicard"


In [ ]:
with open("jsons/pr/Communique FY23.json") as f:
    pr23 = json.load(f)
region23 = pd.DataFrame(pr23["region"])
region23["company name"] = "23-PernodRicard"
finance23 = pd.DataFrame(pr23["finance"])
finance23["company name"] = "23-PernodRicard"
free_cash_flow_debt23 = pd.DataFrame(pr23["free_cash_flow_debt"])
free_cash_flow_debt23["company name"] = "23-PernodRicard"
sales_drinks23 = pd.DataFrame(pr23["sales_drinks"])
sales_drinks23["company name"] = "23-PernodRicard"
results_drinks23 = pd.DataFrame(pr23["results_drinks"])
results_drinks23["company name"] = "23-PernodRicard"
corporate_information23 = pd.DataFrame(pr23["corporate"])
corporate_information23["company name"] = "23-PernodRicard"
social_dei23 = pd.DataFrame(pr23["social_dei"])
social_dei23["company name"] = "23-PernodRicard"

In [ ]:
with open("jsons/pr/pernodFY22.json") as f:
    pr22 = json.load(f)
region22 = pd.DataFrame(pr22["region"])
region22["company name"] = "22-PernodRicard"
finance22 = pd.DataFrame(pr22["finance"])
finance22["company name"] = "22-PernodRicard"
free_cash_flow_debt22 = pd.DataFrame(pr22["free_cash_flow_debt"])
free_cash_flow_debt22["company name"] = "22-PernodRicard"
sales_drinks22 = pd.DataFrame(pr22["sales_drinks"])
sales_drinks22["company name"] = "22-PernodRicard"
results_drinks22 = pd.DataFrame(pr22["results_drinks"])
results_drinks22["company name"] = "22-PernodRicard"
corporate_information22 = pd.DataFrame(pr22["corporate"])
corporate_information22["company name"] = "22-PernodRicard"
social_dei22 = pd.DataFrame(pr22["social_dei"])
social_dei22["company name"] = "22-PernodRicard"

In [ ]:
finance_pernod = pd.concat([finance, finance21, finance22, finance23, finance24], ignore_index=True)
free_cash_flow_pernod = pd.concat([free_cash_flow_debt20, free_cash_flow_debt21, free_cash_flow_debt22, free_cash_flow_debt23, free_cash_flow_debt24], ignore_index=True)
sales_drink_pernod = pd.concat([sales_drinks20, sales_drinks21, sales_drinks22, sales_drinks23, sales_drinks24], ignore_index=True)
results_drink_pernod = pd.concat([results_drinks20, results_drinks21, results_drinks22, results_drinks23, results_drinks24], ignore_index=True)
regions_pernod = pd.concat([region20, region21, region22, region23, region24], ignore_index=True)
corporate_information_pernod = pd.concat([corporate_information20, corporate_information21, corporate_information22, corporate_information23, corporate_information24], ignore_index=True)
social_dei_pernod = pd.concat([social_dei20, social_dei21, social_dei22, social_dei23, social_dei24], ignore_index=True)